# Module 10: Machine Learning with Scikit-Learn
# Lab 04: Unsupervised Learning

**Prepared by Information Tech Consultants Ltd**

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
- [ ] Apply K-Means, DBSCAN, and Hierarchical Clustering to group data
- [ ] Use PCA to reduce dimensionality and visualise high-dimensional data
- [ ] Choose the right number of clusters using the Elbow method and Silhouette Score
- [ ] Explain the difference between supervised and unsupervised learning

**⏱ Estimated Time:** 75 minutes  
**📋 Prerequisites:** Module 10 Labs 01–03

In [ ]:
# ============================================================
# 📦 Environment Setup — Run this cell first!
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler

print("✅ All packages imported successfully!")

In [ ]:
# ============================================================
# 🛠 Helper Functions (run once, use throughout)
# ============================================================

from IPython.display import HTML, display

def info_box(title, content, color="#0092D6", bg="#E3F2FD"):
    """Display a styled information callout box."""
    display(HTML(f"""
    <div style="background:{bg};padding:15px;border-left:5px solid {color};
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>💡 {title}</strong><br>{content}</div>"""))

def warning_box(title, content):
    """Display a warning callout box."""
    display(HTML(f"""
    <div style="background:#FFF3E0;padding:15px;border-left:5px solid #FF9800;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>⚠️ {title}</strong><br>{content}</div>"""))

def interview_box(question, key_points):
    """Display an interview question callout box."""
    display(HTML(f"""
    <div style="background:#F3E5F5;padding:15px;border-left:5px solid #9C27B0;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🎯 Interview Question</strong><br><em>"{question}"</em><br><br>
    <strong>Key Points:</strong> {key_points}</div>"""))

def success_box(content):
    """Display a success/best practice box."""
    display(HTML(f"""
    <div style="background:#E8F5E9;padding:15px;border-left:5px solid #4CAF50;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>✅ Best Practice</strong><br>{content}</div>"""))

def exercise_header(num, title, difficulty="⭐"):
    """Display a formatted exercise header."""
    display(HTML(f"""
    <div style="background:#E8EAF6;padding:15px;border-left:5px solid #0092D6;
    margin:15px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🏋️ Exercise {num}: {title}</strong> | Difficulty: {difficulty}</div>"""))

def draw_pipeline(steps, arrow="→"):
    """Draw a simple pipeline flow diagram."""
    flow = f" {arrow} ".join([f"[{s}]" for s in steps])
    display(HTML(f"""
    <div style="background:#F5F5F5;padding:20px;border-radius:8px;
    text-align:center;font-family:monospace;font-size:16px;margin:10px 0;">
    {flow}</div>"""))

print("✅ Helper functions loaded!")

## 🚀 Complete Working Example

Let's cluster some data using K-Means, visualise it, and then reduce dimensions with PCA — all in one go.

In [ ]:
# ============================================================
# 🚀 COMPLETE WORKING EXAMPLE — Run me first!
# ============================================================

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Generate sample data with 4 natural groups
X_blobs, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)

# Step 1: Cluster with K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_blobs)

# Step 2: Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(X_blobs[:, 0], X_blobs[:, 1], c=y_true, cmap="Set2", s=30, alpha=0.7)
axes[0].set_title("True Clusters", fontsize=14, fontweight="bold")

axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=labels, cmap="Set2", s=30, alpha=0.7)
axes[1].scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
               c="red", marker="X", s=200, edgecolors="black", linewidths=2, label="Centroids")
axes[1].set_title("K-Means Clusters", fontsize=14, fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.show()
print("🎉 K-Means correctly found the 4 clusters!")

---
## 📖 Section 1: K-Means Clustering

**What:** K-Means groups data points into *k* clusters by finding the best centre (centroid) for each cluster and assigning each point to the nearest centre.

**Why it matters:** It's the most widely used clustering algorithm. You'll use it for customer segmentation, image compression, anomaly detection, and more.

**Analogy:** Imagine dropping 3 pins randomly on a map of customer locations. Each pin "attracts" the nearest customers. Then you move each pin to the centre of its group. Repeat until the pins stop moving. That's K-Means!

In [ ]:
# How to choose the right number of clusters? Use the Elbow Method!
from sklearn.cluster import KMeans

inertias = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_blobs)
    inertias.append(km.inertia_)  # inertia = sum of squared distances to centroids

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertias, "o-", color="#0092D6", linewidth=2, markersize=8)
plt.xlabel("Number of Clusters (k)", fontsize=12)
plt.ylabel("Inertia (within-cluster sum of squares)", fontsize=12)
plt.title("Elbow Method — Finding the Optimal k", fontsize=14, fontweight="bold")
plt.axvline(x=4, color="red", linestyle="--", alpha=0.5, label="Elbow at k=4")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

info_box(
    "The Elbow Method",
    "Plot inertia vs k. Look for the 'elbow' — the point where adding more clusters "
    "gives diminishing returns. Here, the elbow is clearly at k=4."
)

In [ ]:
# Silhouette Score: another way to evaluate cluster quality
from sklearn.metrics import silhouette_score

print(f"{'k':<5} {'Silhouette Score':>18}")
print("=" * 26)

for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_blobs)
    sil = silhouette_score(X_blobs, labels)
    bar = "█" * int(sil * 30)
    print(f"{k:<5} {sil:18.4f}  {bar}")

info_box(
    "Silhouette Score",
    "Ranges from -1 to 1.<br>"
    "<b>Close to 1</b>: Well-clustered, points are far from other clusters.<br>"
    "<b>Close to 0</b>: Points are on the boundary between clusters.<br>"
    "<b>Negative</b>: Points might be in the wrong cluster."
)

---
## 📖 Section 2: DBSCAN — Density-Based Clustering

**What:** DBSCAN finds clusters based on density — areas where points are packed closely together. It can discover clusters of arbitrary shape and automatically detect outliers.

**Analogy:** Imagine shining a torch on a crowd of people. Groups standing close together form clusters. People standing alone in the dark are outliers.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.datasets import make_moons

# Generate crescent-shaped data (K-Means will struggle here!)
X_moons, y_moons = make_moons(n_samples=300, noise=0.1, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# K-Means (fails on non-spherical shapes)
km = KMeans(n_clusters=2, random_state=42, n_init=10)
km_labels = km.fit_predict(X_moons)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km_labels, cmap="Set2", s=30)
axes[0].set_title("K-Means (fails!)", fontsize=13, fontweight="bold")

# DBSCAN (handles non-spherical shapes)
db = DBSCAN(eps=0.2, min_samples=5)
db_labels = db.fit_predict(X_moons)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db_labels, cmap="Set2", s=30)
axes[1].set_title("DBSCAN (works!)", fontsize=13, fontweight="bold")

# True labels
axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="Set2", s=30)
axes[2].set_title("True Labels", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

info_box(
    "DBSCAN Parameters",
    "<b>eps</b>: Maximum distance between two points to be considered neighbours.<br>"
    "<b>min_samples</b>: Minimum number of points to form a dense region (cluster).<br>"
    "Points not in any cluster are labelled as <b>-1 (noise/outliers)</b>."
)

---
## 📖 Section 3: Hierarchical Clustering

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage

# Use a small subset for the dendrogram (so it's readable)
X_small = X_blobs[:50]

# Create the linkage matrix for the dendrogram
linkage_matrix = linkage(X_small, method='ward')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dendrogram
dendrogram(linkage_matrix, ax=axes[0], leaf_font_size=8)
axes[0].set_title("Dendrogram", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Sample Index")
axes[0].set_ylabel("Distance")

# Agglomerative Clustering result
agg = AgglomerativeClustering(n_clusters=4)
agg_labels = agg.fit_predict(X_blobs)
axes[1].scatter(X_blobs[:, 0], X_blobs[:, 1], c=agg_labels, cmap="Set2", s=30, alpha=0.7)
axes[1].set_title("Agglomerative Clustering (k=4)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

interview_box(
    "Compare K-Means, DBSCAN, and Hierarchical Clustering.",
    "<b>K-Means:</b> Fast, needs k specified, assumes spherical clusters.<br>"
    "<b>DBSCAN:</b> Finds arbitrary shapes, detects outliers, no k needed, but eps is sensitive.<br>"
    "<b>Hierarchical:</b> Produces a dendrogram showing relationships, no k needed upfront, but slow on large data."
)

---
## 📖 Section 4: PCA — Dimensionality Reduction

**What:** PCA (Principal Component Analysis) reduces the number of features while keeping as much information as possible.

**Why it matters:** Helps with visualisation, speeds up training, and can reduce noise.

**Analogy:** Imagine taking a photo of a 3D object from the best angle that captures the most detail. PCA finds that "best angle" for your data.

In [ ]:
from sklearn.decomposition import PCA

# Load the Iris dataset (4 features — hard to visualise in 4D!)
iris = load_iris(as_frame=True)
X_iris = iris.frame.drop("target", axis=1)
y_iris = iris.frame["target"]

# Scale first
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Reduce from 4 dimensions to 2
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_iris_scaled)

# How much information did we keep?
print("Explained Variance Ratio:")
for i, var in enumerate(pca.explained_variance_ratio_):
    bar = "█" * int(var * 50)
    print(f"  PC{i+1}: {var:.4f} ({var*100:.1f}%)  {bar}")
print(f"  Total: {sum(pca.explained_variance_ratio_):.4f} ({sum(pca.explained_variance_ratio_)*100:.1f}%)")

In [ ]:
# Visualise the 4D Iris data in 2D using PCA
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# PCA 2D view
scatter = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_iris, cmap="Set2", s=40, alpha=0.8)
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)", fontsize=12)
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)", fontsize=12)
axes[0].set_title("Iris Dataset — PCA (2 Components)", fontsize=14, fontweight="bold")
axes[0].legend(*scatter.legend_elements(), title="Species",
               labels=iris.target_names)

# How many components to keep?
pca_full = PCA()
pca_full.fit(X_iris_scaled)
cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumulative_var)+1), cumulative_var, "o-", color="#0092D6", linewidth=2)
axes[1].axhline(y=0.95, color="red", linestyle="--", alpha=0.5, label="95% threshold")
axes[1].set_xlabel("Number of Components", fontsize=12)
axes[1].set_ylabel("Cumulative Explained Variance", fontsize=12)
axes[1].set_title("PCA — Variance Explained", fontsize=14, fontweight="bold")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

success_box(
    "A common rule of thumb: keep enough components to explain <b>95%</b> of the variance. "
    "For the Iris dataset, just 2 components capture about 96% of the information!"
)

---
## 🏋️ Exercises

In [ ]:
exercise_header(1, "Find the Elbow", "⭐")

# Run this and identify the optimal number of clusters
from sklearn.datasets import make_blobs

X_ex, _ = make_blobs(n_samples=500, centers=6, cluster_std=1.0, random_state=99)

inertias = []
for k in range(1, 12):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_ex)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 12), inertias, "o-", color="#0092D6", linewidth=2, markersize=8)
plt.xlabel("k"); plt.ylabel("Inertia")
plt.title("Elbow Method", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)
plt.show()

# ❓ Where is the elbow? What is the optimal k?

In [ ]:
exercise_header(2, "DBSCAN vs K-Means", "⭐⭐")

# TODO: Generate crescent-shaped data using make_moons (n_samples=500, noise=0.15)
# Apply both K-Means (k=2) and DBSCAN (try eps=0.2, min_samples=5)
# Plot both results side by side

from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, DBSCAN

# YOUR CODE HERE:
# X_moons, y_moons = make_moons(...)
# km = KMeans(...)
# db = DBSCAN(...)
# Create a figure with 2 subplots showing each result

In [ ]:
exercise_header(3, "PCA + Clustering Pipeline", "⭐⭐⭐")

# Challenge: Take the wine dataset (13 features), reduce to 2 with PCA,
# then apply K-Means. Visualise the clusters in PCA space.

from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.frame.drop("target", axis=1)
y_wine = wine.frame["target"]

# YOUR CODE HERE:
# 1. Scale the data
# 2. Apply PCA (n_components=2)
# 3. Apply K-Means (k=3, since wine has 3 classes)
# 4. Create a scatter plot showing clusters in PCA space
# 5. Compare cluster labels with actual wine classes

---
## 📋 Solutions

<details>
<summary>Click to expand Exercise 2 solution</summary>

```python
X_moons, y_moons = make_moons(n_samples=500, noise=0.15, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
km = KMeans(n_clusters=2, random_state=42, n_init=10)
axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=km.fit_predict(X_moons), cmap="Set2", s=20)
axes[0].set_title("K-Means")

db = DBSCAN(eps=0.2, min_samples=5)
axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=db.fit_predict(X_moons), cmap="Set2", s=20)
axes[1].set_title("DBSCAN")
plt.tight_layout(); plt.show()
```

</details>

<details>
<summary>Click to expand Exercise 3 solution</summary>

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_wine)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = km.fit_predict(X_pca)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=km_labels, cmap="Set2", s=30)
axes[0].set_title("K-Means Clusters")
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=y_wine, cmap="Set2", s=30)
axes[1].set_title("True Wine Classes")
plt.tight_layout(); plt.show()
```

</details>

---
## 🎯 Key Takeaways

1. **Unsupervised learning** finds patterns without labels — used for clustering and dimensionality reduction
2. **K-Means** is fast and simple but needs k specified and assumes spherical clusters
3. **DBSCAN** discovers arbitrary shapes and outliers — great for real-world messy data
4. **PCA** reduces dimensions while preserving information — essential for visualisation and preprocessing
5. **Use the Elbow Method or Silhouette Score** to choose the right number of clusters

## ✅ Self-Assessment Checklist
- [ ] I can apply K-Means and find the optimal k
- [ ] I understand when DBSCAN is better than K-Means
- [ ] I can use PCA to reduce dimensions and visualise data
- [ ] I know how to interpret explained variance ratios

## 📚 Next Steps
- **Next Lab:** Module 10 Lab 05 — Model Tuning & Pipelines
- **Practice:** Apply clustering to a real-world dataset (try the Mall Customers dataset on Kaggle)

---
*Prepared by Information Tech Consultants Ltd*